## This is some code to generate biomass estimates from UAV data

Segmentation parameters below are the best-scoring combo from the parameter sweep
(see `Parameter_Sweep_Analysis/`), once the sweep scoring also accounted for
under/over-segmentation (total segmented tree count vs. the ~592-tree TreePlotter
Data census), not just match count/MAE within footprints:

- **0.4m CHM** (overall best, count-fidelity adj. score 32.69): 3x3 kernel, fixed ws=3.0
  -- within +2.2% of the 592-tree reference count.
- **0.25m CHM** (best within that sweep, score 30.22): 5x5 kernel, variable ws
  (`var_high`: intercept=2.0, slope=0.1, cap=6) -- within +7.4% of the reference count.

The segmentation cell below runs **both** CHMs in one pass: it prompts for the 0.4m
CHM file, then the 0.25m CHM file, then the (shared) GEDI footprints shapefile, and
writes separate outputs for each resolution to `../outputs/` --
`tree_heights_04m.csv`/`treetops_04m.gpkg` and `tree_heights_025m.csv`/`treetops_025m.gpkg`
-- rather than overwriting one shared `tree_heights.csv`. Point
`scripts/compare_heights_within_footprints.py` / `scripts/compare_segmentation_to_treeplotter.py` at
whichever of the two you want to analyze (pass `04m` or `025m` as the CLI arg).

This replaces the earlier v1 (3x3 kernel, fixed ws=2.5), v2 (5x5 kernel, variable ws
intercept=2.0/slope=0.1/cap=6), and the first count-fidelity-blind sweep pick (3x3
kernel, var_low), which all scored lower once count fidelity was factored in.

1. Perform Tree segmentation to get separate all the individual trees
    a. across the entire plot
    b. within GEDI footprints only

2. Retreive and Store Tree height and canopy cover information for each tree
    a. across the entire plot
    b. within the GEDI footprints

2a. get tree specific height comparison info for trees vs ground measurements

3. Find algorithms to interpolate tree info to biomass

4. generate biomass estimates


In [1]:
# imports 
# may need to run this in command line because packages are not available in R 4.6.1
#install.packages(c('terra', 'sf', 'lidR', 'rlas', 'dplyr', 'geojsonio'), repos='https://cran.r-project.org')

library(terra)
library(sf)
library(lidR)
library(dplyr)

terra 1.9.34

Linking to GEOS 3.14.1, GDAL 3.12.1, PROJ 9.7.1; sf_use_s2() is TRUE


Attaching package: 'lidR'


The following object is masked from 'package:sf':

    st_concave_hull


The following object is masked from 'package:terra':

    watershed



Attaching package: 'dplyr'


The following objects are masked from 'package:terra':

    intersect, union


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union




In [ ]:

#set some color palettes
col <- height.colors(50)
col1 <- pastel.colors(900)

# Best-scoring combo per CHM resolution, from the parameter sweep
# (see Parameter_Sweep_Analysis/), count-fidelity adjusted against the
# 592-tree TreePlotter Data reference.
chm_configs <- list(
  list(
    label = "0.4m",
    suffix = "04m",
    kernel = matrix(1, 3, 3),
    ws_algorithm = 3.0
  ),
  list(
    label = "0.25m",
    suffix = "025m",
    kernel = matrix(1, 5, 5),
    ws_algorithm = function(x) {
      y <- 2.0 + 0.1 * x
      y[x < 2] <- 2.0
      y[y > 6] <- 6.0
      return(y)
    }
  )
)

# Outputs are written to ../outputs/ (relative to this notebook's location
# in notebooks/), which is gitignored since these are regenerated results.
output_dir <- file.path(dirname(getwd()), "outputs")
if (!dir.exists(output_dir)) dir.create(output_dir, recursive = TRUE)

# Load GEDI footprints once -- shared across both CHM resolutions.
message("Please select your GEDI footprints clipped shapefile.")
footprint_path <- file.choose()
footprints <- st_read(footprint_path)

results <- list()

for (cfg in chm_configs) {
  cat("\n=== Processing", cfg$label, "CHM ===\n")

  message("Please select your ", cfg$label, " clipped CHM file in the file browser.")
  chm_clipped_path <- file.choose()
  cat("Selected file:", chm_clipped_path, "\n")
  chm_clipped <- rast(chm_clipped_path)

  # Generate kernel and smooth chm
  schm <- terra::focal(x = chm_clipped, w = cfg$kernel, fun = median, na.rm = TRUE)
  plot(schm, col = col, main = paste(cfg$label, "smoothed CHM"))

  # Detect trees
  ttops <- locate_trees(las = schm, algorithm = lmf(ws = cfg$ws_algorithm))
  print(ttops)

  # Check and align CRS
  if (st_crs(footprints) != st_crs(ttops)) {
    message("CRS mismatch detected — reprojecting footprints to match CHM...")
    footprints_aligned <- st_transform(footprints, st_crs(ttops))
  } else {
    message("CRS match confirmed.")
    footprints_aligned <- footprints
  }

  # Filter to only trees within footprints
  ttops_in_footprints <- st_intersection(ttops, footprints_aligned)

  # Count trees per footprint
  tree_counts <- ttops_in_footprints |>
    st_drop_geometry() |>
    group_by(reading_order_id) |>
    summarise(tree_count = n())
  print(tree_counts)

  # Store the Height data for each segmented tree
  # Extract ttops Z coordinate where height data is stored: ttops = treeID, geometry (X, Y, Z)
  ttops$height <- st_coordinates(ttops)[, "Z"]

  # Save as a shapefile or geopackage to keep geometry + attributes together
  gpkg_out <- file.path(output_dir, paste0("treetops_", cfg$suffix, ".gpkg"))
  st_write(ttops, gpkg_out, delete_dsn = TRUE)

  # Recreate clean data frame with tree ID, location, and height to load into Excel sheet
  tree_data <- data.frame(
    treeID   = ttops$treeID,
    x        = st_coordinates(ttops)[, "X"],
    y        = st_coordinates(ttops)[, "Y"],
    height_m = st_coordinates(ttops)[, "Z"]
  )
  csv_out <- file.path(output_dir, paste0("tree_heights_", cfg$suffix, ".csv"))
  write.csv(tree_data, csv_out, row.names = FALSE)
  cat("Wrote", csv_out, "and", gpkg_out, "-", nrow(tree_data), "trees\n")

  # Plot
  plot(chm_clipped, col = col, main = paste("Detected Trees within Footprints --", cfg$label))
  plot(st_geometry(footprints_aligned), border = "black", add = TRUE)
  plot(st_geometry(ttops_in_footprints), add = TRUE, pch = 3, col = "red", cex = 0.5)

  results[[cfg$suffix]] <- tree_data
}

cat("\nDone. Outputs written to", output_dir, "\n")
